### Cell 1: Importing Required Libraries and Preparing File Paths

In this cell, I am importing **Python libraries** like `os`, `csv`, and `json` that help in handling files and structured data.  

This cell will also set up paths for input and output directories, where processed files will be stored later.  


In [1]:
# Cell 1: Importing DataSet
import os ,csv ,json # Importing Python libraries
from datetime import datetime 

CSV_FILE ="acw_user_data.csv"# Sir Please Change File Name if Different
OUTPUT_DIR ="acw_outputs"# Setting output file directory
os .makedirs (OUTPUT_DIR ,exist_ok =True )# Create the output folder if it doesn’t already exist

# Helper Functions
def clean_str (x ):return (x or "").strip ()
def to_int (x ,default =None ):# Function to convert a value to an integer
    try :return int (float (clean_str (x )))
    except :return default 
def to_float (x ,default =None ):# Function to convert a value to a float
    try :return float (clean_str (x ))
    except :return default 
def is_truthy (x ):# Function to convert yes/no or true/false type text into a Boolean (True/False)
    return clean_str (x ).lower ()in {"yes","true","1","y","t"}
def write_json (obj ,name ):# Function to write any Python object to a JSON file
    path =os .path .join (OUTPUT_DIR ,name )
    with open (path ,"w",encoding ="utf-8")as file_handle :
        json .dump (obj ,file_handle ,indent =2 ,ensure_ascii =False )
    print ("Wrote:",path )


### Cell 2: Reading Data Files from Directory

Here, I am reading the CSV files that will be used for data processing.  

 It also extracts **column headers** and **rows** to understand the file structure.
 
- After reading, it **prints out 5 rows and all headers** so I can confirm that the file path and data format are correct.

We are verifying that the data is being loaded properly here.  

The main idea here is to **access the data files** and **make it ready for further cleaning and transformation**.


In [2]:
# Cell 2: Reading file into memory & then inspect
import csv 

CSV_FILE ="acw_user_data.csv"
with open (CSV_FILE ,newline ='',encoding ='utf-8')as file_handle :
    rows =list (csv .reader (file_handle ))# reads entire file into memory

if not rows :
    raise RuntimeError ("CSV appears empty.")

headers =rows [0 ]

# Print all column headers with their index number to verify structure
print ("Headers (index : repr(header)):")
for i ,h in enumerate (headers ):
    print (i ,repr (h ))

    # Print the first 5 data rows to inspect raw values
print ("\nFirst 5 rows (raw):")
for i ,row in enumerate (rows [1 :6 ],start =1 ):
    print (i ,row )


Headers (index : repr(header)):
0 'Address Street'
1 'Address City'
2 'Address Postcode'
3 'Age (Years)'
4 'Distance Commuted to Work (Km)'
5 'Employer Company'
6 'Credit Card Start Date'
7 'Credit Card Expiry Date'
8 'Credit Card Number'
9 'Credit Card CVV'
10 'Dependants'
11 'First Name'
12 'Bank IBAN'
13 'Last Name'
14 'Marital Status'
15 'Yearly Pension (Dollar)'
16 'Retired'
17 'Yearly Salary (Dollar)'
18 'Sex'
19 'Vehicle Make'
20 'Vehicle Model'
21 'Vehicle Year'
22 'Vehicle Type'

First 5 rows (raw):
1 ['70 Lydia isle', 'Lake Conor', 'S71 7XZ', '89', '0', 'N/A', '08/18', '11/27', '676373692463', '875', '3', 'Kieran', 'GB62PQKB71416034141571', 'Wilson', 'married or civil partner', '7257', 'True', '72838', 'Male', 'Hyundai', 'Bonneville', '2009', 'Pickup']
2 ['00 Wheeler wells', 'Chapmanton', 'L2 7BT', '46', '13.72', 'Begum-Williams', '08/12', '11/26', '4529436854129855', '583', '1', 'Jonathan', 'GB37UMCO54540228728019', 'Thomas', 'married or civil partner', '0', 'False', '54016'

### Cell 3: Processing and Cleaning CSV Data for Further Use

We are performing data cleaning on the raw CSV data we read earlier.  

- Provided **Variable names** to Header of the file such as `H_AGE`, `H_COMM`, `H_DEP`, and `employed`.
- We will be checking missing or invalid values and correcting them.
- Code will ignore placeholders like **"N/A"** for handling employer.
- Will store the final cleaned data in a list of dictionaries.
- It will print a short summary showing how many rows were processed and how many records were fixed.  

These all steps ensures that the data is consistent, complete, and formatted properly before analysis.


In [3]:
# Cell 3 — Core Data Preprocessing
import csv 

# Lists to store processed records and special groups
processed_records =[]
retired =[]
employed =[]
expired_cards =[]
dependant_fixed_rows =[]

# Helper Functions
def clean_str (x ):return (x or "").strip ()
def to_int (x ,default =None ):
    try :return int (float (clean_str (x )))
    except :return default 
def to_float (x ,default =None ):
    try :return float (clean_str (x ))
    except :return default 
def is_truthy (x ):
    return clean_str (x ).lower ()in {"yes","true","1","y","t"}

    # Header name constants (using these to access columns safely)
dependants_col ="Dependants"
age_col ="Age (Years)"
commute_col ="Distance Commuted to Work (Km)"
employer_col ="Employer Company"
credit_card_start_col ="Credit Card Start Date"
credit_card_expiry_col ="Credit Card Expiry Date"
credit_card_number_col ="Credit Card Number"
cvv_col ="Credit Card CVV"
iban_col ="Bank IBAN"
retired_col ="Retired"
salary_col ="Yearly Salary (Dollar)"
first_name_col ="First Name"
last_name_col ="Last Name"
address_street_col ="Address Street"
address_city_col ="Address City"
address_postcode_col ="Address Postcode"
vehicle_make_col ="Vehicle Make"
vehicle_model_col ="Vehicle Model"
vehicle_year_col ="Vehicle Year"
vehicle_type_col ="Vehicle Type"
marital_status_col ="Marital Status"

# Date parsing helpers like parse mm/yy to (m, yyyy)
def parse_mm_yy (txt ):
    t =clean_str (txt )
    if "/"not in t :return None 
    a ,b =t .split ("/")
    try :
        return (int (a ),2000 +int (b ))
    except :
        return None 

def years_between (a ,b ):
    if not a or not b :return None 
    am ,ay =a ;bm ,by =b 
    months =(by -ay )*12 +(bm -am )
    return months /12.0 

    # Main loop: read CSV and process each row
with open ("acw_user_data.csv",newline ='',encoding ='utf-8')as file_handle :
    csv_reader =csv .DictReader (file_handle )
    for row_index ,row in enumerate (csv_reader ):
    # safe getters
        def get_field (h ):
            if h in row :return row [h ]
            for k in row :
                if k and k .strip ().lower ()==h .strip ().lower ():
                    return row [k ]
            return ""

        first =clean_str (get_field (first_name_col ));last =clean_str (get_field (last_name_col ))
        age =to_int (get_field (age_col ),None )
        salary =to_float (get_field (salary_col ),0.0 )

        # Dependants: if blank, set 0 and record index for verification
        deps_raw =get_field (dependants_col )
        if clean_str (deps_raw )=="":
            dependants =0 
            dependant_fixed_rows .append (row_index )
        else :
            dependants =to_int (deps_raw ,0 )

            # Vehicle details grouped in a small dictionary
        vehicle ={
        "make":clean_str (get_field (vehicle_make_col )),
        "model":clean_str (get_field (vehicle_model_col )),
        "year":to_int (get_field (vehicle_year_col ),None ),
        "type":clean_str (get_field (vehicle_type_col ))
        }

        # Credit card info grouped
        credit_card ={
        "start_date":clean_str (get_field (credit_card_start_col )),
        "end_date":clean_str (get_field (credit_card_expiry_col )),
        "number":clean_str (get_field (credit_card_number_col )),
        "security_code":to_int (get_field (cvv_col ),None ),
        "iban":clean_str (get_field (iban_col ))
        }

        # Address grouped
        address ={
        "address_line":clean_str (get_field (address_street_col )),
        "city":clean_str (get_field (address_city_col )),
        "postcode":clean_str (get_field (address_postcode_col ))
        }

        # Retirement boolean
        retired_flag =is_truthy (get_field (retired_col ))

        # Handling Employers here by ignoring placeholders like "N/A"
        _raw_emp =clean_str (get_field (employer_col ))
        if _raw_emp .lower ()in {"","n/a","na","none","-","0"}:
            employer_val =""
        else :
            employer_val =_raw_emp 

        commute_val =to_float (get_field (commute_col ),None )
        marital =clean_str (get_field (marital_status_col ))

        # Build final person dictionary
        person ={
        "first_name":first ,
        "last_name":last ,
        "age":age ,
        "salary":salary ,
        "dependants":dependants ,
        "vehicle":vehicle ,
        "credit_card":credit_card ,
        "address":address ,
        "employer":employer_val ,
        "retired":retired_flag ,
        "commute_km":commute_val ,
        "marital_status":marital 
        }

        # Append to main list and group lists
        processed_records .append (person )
        if retired_flag :
            retired .append (person )
        if employer_val :
            employed .append (person )

            # flag credit cards >10 years
        start_p =parse_mm_yy (credit_card ["start_date"])
        end_p =parse_mm_yy (credit_card ["end_date"])
        yrs =years_between (start_p ,end_p )
        if yrs is not None and yrs >10 :
            expired_cards .append (person )

            # Summary prints for verification
print ("Rows processed:",len (processed_records ))
print ("Dependants auto-fixed (count):",len (dependant_fixed_rows ))
print ("Sample fixed indices (first 20):",dependant_fixed_rows [:20 ])
print ("Retired:",len (retired ),"Employed:",len (employed ),"Remove_ccard:",len (expired_cards ))
print ("Sample processed record (first):")
import pprint ;pprint .pprint (processed_records [0 ])


Rows processed: 1000
Dependants auto-fixed (count): 19
Sample fixed indices (first 20): [21, 109, 179, 205, 270, 272, 274, 358, 460, 468, 579, 636, 679, 725, 822, 865, 917, 931, 983]
Retired: 246 Employed: 754 Remove_ccard: 252
Sample processed record (first):
{'address': {'address_line': '70 Lydia isle',
             'city': 'Lake Conor',
             'postcode': 'S71 7XZ'},
 'age': 89,
 'commute_km': 0.0,
 'credit_card': {'end_date': '11/27',
                 'iban': 'GB62PQKB71416034141571',
                 'number': '676373692463',
                 'security_code': 875,
                 'start_date': '08/18'},
 'dependants': 3,
 'employer': '',
 'first_name': 'Kieran',
 'last_name': 'Wilson',
 'marital_status': 'married or civil partner',
 'retired': True,
 'salary': 72838.0,
 'vehicle': {'make': 'Hyundai',
             'model': 'Bonneville',
             'type': 'Pickup',
             'year': 2009}}


### Cell 4: Writing Processed Data into JSON Files

In this cell, I created a subfolder called **`acw_outputs/json`** and defining a function named `write_json()`.

This function saves cleaned or processed data in the **JSON format** with proper indentation.  
When executed, it prints the confirmation of each file it writes.


In [4]:
# Cell 4: Writing outputs to acw_outputs/json/
import os ,json 

# Create main and subfolders
base_dir ="acw_outputs"
json_dir =os .path .join (base_dir ,"json")
os .makedirs (json_dir ,exist_ok =True )

# Define a helper function to save any Python object as a JSON file
def write_json (obj ,name ):
    """Save JSON into acw_outputs/json/"""
    path =os .path .join (json_dir ,name )
    with open (path ,"w",encoding ="utf-8")as file_handle :
        json .dump (obj ,file_handle ,indent =2 ,ensure_ascii =False )
    print ("Wrote:",path )

    # Save all JSON outputs
write_json (processed_records ,"processed.json")
write_json (retired ,"retired.json")
write_json (employed ,"employed.json")
write_json (expired_cards ,"remove_ccard.json")


Wrote: acw_outputs\json\processed.json
Wrote: acw_outputs\json\retired.json
Wrote: acw_outputs\json\employed.json
Wrote: acw_outputs\json\remove_ccard.json


### Cell 5: Adding a Salary-to-Commute Ratio and Sorting Data

This cell defines a function called `salary_commute(person)` which calculates a **salary-to-commute ratio** for each record.  
The function divides a person’s salary by their commute distance (`salary / commute_km`) to measure how efficient the income is relative to travel.

If a person doesn’t have commute data or has an invalid value, the code safely handles it.

After adding this new column, the program sorts all records by `salary_commute` value, making it easier to identify which individuals have the best or worst ratios.

It also saves the results into `acw_outputs/json/commute.json` and prints the three lowest salary-to-commute ratios.  


In [5]:
# Cell 5: compute salary_commute, sort, and save to acw_outputs/json/
def salary_commute (person ):# Function to calculate salary-to-commute ratio for each person
    sal =person .get ("salary")or 0.0 
    c =person .get ("commute_km")
    try :
        if c is None or c <=1 :
            return float (sal )
        return float (sal )/float (c )
    except :
        return float (sal )

        # Add a new column "salary_commute" for every record in the processed dataset
for person_obj in processed_records :
    person_obj ["salary_commute"]=salary_commute (person_obj )

    # Sort the processed list by the new salary_commute value (lowest first)
processed_sorted =sorted (processed_records ,key =lambda x :x .get ("salary_commute",float ("inf")))

# Save into json subfolder
import json ,os 
json_dir =os .path .join ("acw_outputs","json")
os .makedirs (json_dir ,exist_ok =True )

commute_path =os .path .join (json_dir ,"commute.json")
with open (commute_path ,"w",encoding ="utf-8")as file_handle :
    json .dump (processed_sorted ,file_handle ,indent =2 ,ensure_ascii =False )

    # Print confirmation and a sample of results
print (f"Wrote: {commute_path}")
print ("Sample top 3 (lowest salary_commute):")
for person_obj in processed_sorted [:3 ]:
    print (person_obj ["first_name"],person_obj ["last_name"],person_obj ["salary_commute"])


Wrote: acw_outputs\json\commute.json
Sample top 3 (lowest salary_commute):
Graeme Jackson 3088.04347826087
Janet Quinn 3090.070921985816
Peter Burton 3090.5017921146955


### Note: Completion of CSV-Based Preprocessing

Up to this point, all the preprocessing has been done **strictly using the CSV library only**, as required in the task instructions.  
I have used CSV functions to read, clean, and process the raw data without relying on external libraries like Pandas.  

The data has now been cleaned, categorized, and saved into structured JSON files, which completes the **CSV-based preprocessing phase**.  
From the next cell onward, I will start using **Pandas** for analysis and visualization.









**Starting Data Visualisation Next**

### Cell 6: Using Pandas to Calculate Mean Salary and Median Age

In this cell, I am switching to the **Pandas library**.  
The code loads the file **acw_user_data.csv** into a DataFrame and changes the salary and age columns into numbers so they can be used for calculations.

Then we calculate:
- **Mean Salary** → The average salary of all users.
- **Median Age** → The middle value of the age distribution.

Saves Output in this directory **acw_outputs\json\stats_summary.json**

Then print the results for easy checking.


In [6]:
# Cell 6 : Calculate the mean salary & median age Using Pandas
import pandas as pd ,os ,json 

# Read the dataset
user_df =pd .read_csv ("acw_user_data.csv")
user_df .columns =[c .strip ()for c in user_df .columns ]

# Convert 'Yearly Salary (Dollar)' and 'Age (Years)' columns to numeric types
user_df ['Yearly Salary (Dollar)']=pd .to_numeric (user_df ['Yearly Salary (Dollar)'],errors ='coerce')
user_df ['Age (Years)']=pd .to_numeric (user_df ['Age (Years)'],errors ='coerce')

# Calculate the mean salary (average) and median age (middle value)
mean_salary =user_df ['Yearly Salary (Dollar)'].mean ()
median_age =user_df ['Age (Years)'].median ()

# Print the calculated values for verification
print ("Mean Salary:",mean_salary )
print ("Median Age:",median_age )

# Save JSON File inside acw_outputs/json/
json_dir =os .path .join ("acw_outputs","json")
os .makedirs (json_dir ,exist_ok =True )

stats_path =os .path .join (json_dir ,"stats_summary.json")
with open (stats_path ,"w",encoding ="utf-8")as file_handle :
    json .dump ({"mean_salary":mean_salary ,"median_age":median_age },file_handle ,indent =2 ,ensure_ascii =False )
print (f"Saved: {stats_path}")


Mean Salary: 57814.078
Median Age: 54.0
Saved: acw_outputs\json\stats_summary.json


### Cell 7: Visualizing Age Distribution and Saving the Plot

In this cell, I use **Seaborn** and **Matplotlib** to create several visualizations that helps understand the data better. 
 
The main goal here is to explore relationships between columns like **Age**, **Salary**, **Dependants**, **Commute Distance**, and **Marital Status** using different types of graphs.

The code automatically saves all the generated charts into the folder **`acw_outputs/png/`**, and Prints confirmation after saving.



### Explanation of Each Plot:
- **age_hist_binwidth5.png** → A histogram showing how ages are distributed in 5-year intervals.  
- **dependants_count.png** → A bar chart showing how many people have 0, 1, 2, or more dependants.  
- **age_by_marital_status.png** → Compares average ages across different marital statuses.  
- **commute_vs_salary.png** → Displays the relationship between commute distance and salary.  
- **age_vs_salary.png** → A scatter plot to see how salary varies with age.  
- **age_vs_salary_by_dependants.png** → Similar scatter plot, but divided by number of dependants.

In [7]:
# Cell 7: create plots (seaborn + matplotlib) — outputs go to acw_outputs/png/
import seaborn as sns ,matplotlib .pyplot as plt ,math ,os 

# Create an output folder for saving all PNG charts
OUT =os .path .join ("acw_outputs","png")
os .makedirs (OUT ,exist_ok =True )# ensure png folder exists

# Age histogram with bin_width 5
age_min =int (user_df ['Age (Years)'].min ())
age_max =int (user_df ['Age (Years)'].max ())
bin_width =5 
num_bins =max (1 ,math .ceil ((age_max -age_min )/bin_width ))
plt .figure ();sns .histplot (user_df ['Age (Years)'].dropna (),bins =num_bins ,color="teal")
plt .title ("Age histogram (bin width=5)")
plt .savefig (os .path .join (OUT ,"age_hist_binwidth5.png"));plt .close ()
print ("Saved:",os .path .join (OUT ,"age_hist_binwidth5.png"))

# Dependants count (fill NA as 0)
dep_col =None 
for c in user_df .columns :
    if c .strip ().lower ().startswith ("depend"):
        dep_col =c ;break 
if dep_col :
    user_df [dep_col ]=pd .to_numeric (user_df [dep_col ],errors ='coerce').fillna (0 ).astype (int )
    plt .figure ();sns .countplot (x =dep_col ,data =user_df ,palette="Set2")
    plt .title ("Dependants counts (NA->0)")
    plt .savefig (os .path .join (OUT ,"dependants_count.png"));plt .close ()
    print ("Saved:",os .path .join (OUT ,"dependants_count.png"))
else :
    print ("Dependants column not found for plotting.")

    # Age by Marital Status
if 'Marital Status'in user_df .columns :
    plt .figure ()
    sns .histplot (data =user_df ,x ='Age (Years)',hue ='Marital Status',
    element ='step',stat ='count',common_norm =False ,palette="Set2")
    plt .title ("Age distribution by Marital Status")
    plt .savefig (os .path .join (OUT ,"age_by_marital_status.png"));plt .close ()
    print ("Saved:",os .path .join (OUT ,"age_by_marital_status.png"))
else :
    print ("Marital Status column not found.")

    # Commute vs Salary (scatter)
comm_col =None 
for c in user_df .columns :
    if "commut"in c .strip ().lower ()or "distance"in c .strip ().lower ():
        comm_col =c ;break 
if comm_col :
    user_df [comm_col ]=pd .to_numeric (user_df [comm_col ],errors ='coerce')
    plt .figure ();sns .scatterplot (data =user_df ,x =comm_col ,y ='Yearly Salary (Dollar)',color="purple")
    plt .title ("Commute vs Salary")
    plt .savefig (os .path .join (OUT ,"commute_vs_salary.png"));plt .close ()
    print ("Saved:",os .path .join (OUT ,"commute_vs_salary.png"))
else :
    print ("Commute column not found for plotting.")

    # Age vs Salary
plt .figure ();sns .scatterplot (data =user_df ,x ='Age (Years)',y ='Yearly Salary (Dollar)',color="darkred")
plt .title ("Age vs Salary")
plt .savefig (os .path .join (OUT ,"age_vs_salary.png"));plt .close ()
print ("Saved:",os .path .join (OUT ,"age_vs_salary.png"))

# Age vs Salary hue=Dependants (if dep_col exists)
if dep_col :
    plt .figure ();sns .scatterplot (data =user_df ,x ='Age (Years)',
    y ='Yearly Salary (Dollar)',hue =dep_col ,palette="Set2")
    plt .title ("Age vs Salary (hue=Dependants)")
    plt .savefig (os .path .join (OUT ,"age_vs_salary_by_dependants.png"),
    bbox_inches ='tight');plt .close ()
    print ("Saved:",os .path .join (OUT ,"age_vs_salary_by_dependants.png"))


Saved: acw_outputs\png\age_hist_binwidth5.png
Saved: acw_outputs\png\dependants_count.png
Saved: acw_outputs\png\age_by_marital_status.png
Saved: acw_outputs\png\commute_vs_salary.png
Saved: acw_outputs\png\age_vs_salary.png
Saved: acw_outputs\png\age_vs_salary_by_dependants.png


### Cell 8: Listing Saved Files and Previewing Processed Data

This cell checks the final output folders to make sure all files were created successfully.  

- Finally, it prints a **preview of the first 3 records** from the processed data to show that the cleaning worked correctly.

This helps verify that both the preprocessing and visualization stages completed successfully and all outputs are organized in their correct folders.

In [8]:
# Cell 8 Quick check of output files and data preview
import os ,json 

base_dir ="acw_outputs"
json_dir =os .path .join (base_dir ,"json")
png_dir =os .path .join (base_dir ,"png")

print ("JSON files in acw_outputs/json/:")
if os .path .exists (json_dir ):
    for file_name in sorted (os .listdir (json_dir )):
        print (" -",file_name )
else :
    print ("  (No JSON folder found)")

print ("\nPNG files in acw_outputs/png/:")
if os .path .exists (png_dir ):
    for file_name in sorted (os .listdir (png_dir )):
        print (" -",file_name )
else :
    print ("  (No PNG folder found)")

    # Preview first few processed records
print ("\nPreview of first 3 processed records:")
for rec in processed_records [:3 ]:
    print (json .dumps (rec ,indent =2 )[:600 ])
    print ("----")


JSON files in acw_outputs/json/:
 - .ipynb_checkpoints
 - commute.json
 - employed.json
 - processed.json
 - remove_ccard.json
 - retired.json
 - stats_summary.json

PNG files in acw_outputs/png/:
 - .ipynb_checkpoints
 - age_by_marital_status.png
 - age_hist_binwidth5.png
 - age_vs_salary.png
 - age_vs_salary_by_dependants.png
 - commute_vs_salary.png
 - dependants_count.png

Preview of first 3 processed records:
{
  "first_name": "Kieran",
  "last_name": "Wilson",
  "age": 89,
  "salary": 72838.0,
  "dependants": 3,
  "vehicle": {
    "make": "Hyundai",
    "model": "Bonneville",
    "year": 2009,
    "type": "Pickup"
  },
  "credit_card": {
    "start_date": "08/18",
    "end_date": "11/27",
    "number": "676373692463",
    "security_code": 875,
    "iban": "GB62PQKB71416034141571"
  },
  "address": {
    "address_line": "70 Lydia isle",
    "city": "Lake Conor",
    "postcode": "S71 7XZ"
  },
  "employer": "",
  "retired": true,
  "commute_km": 0.0,
  "marital_status": "married or